# FAQ And Troubleshooting


## My TMAP has detached branches

Try these in order:

1. increase `kc`
2. increase `n_neighbors`
3. for `jaccard`, increase `n_permutations`
4. check whether your data really is split into separate groups


## My map changes between runs

Set `seed=42`. If you also pass a `LayoutConfig`, set `cfg.deterministic = True` and `cfg.seed = 42`.


## TMAP is slow

- for `jaccard`, try smaller `n_permutations`
- for dense metrics, USearch uses exact search for small datasets and HNSW for larger ones
- reduce `layout_iterations` if layout quality is good enough


## Do cosine and euclidean need extra installation?

No. Dense metrics use USearch from the core install. A plain `python -m pip install .` is enough for the dense backend.


## Why do edges not show in the notebook widget?

Notebook widgets focus on points, colors, and filters. Use HTML export when you want browser rendering with tree edges.


## How do I choose between Jaccard and cosine/euclidean?

- **Jaccard** (default): binary fingerprints, one-hot encodings, sparse binary data. Uses MinHash + LSHForest.
- **Cosine**: dense embeddings (ESM-2, ProtTrans, sentence transformers). Uses USearch.
- **Euclidean**: dense features where absolute magnitude matters. Uses USearch.
- **Precomputed**: you already have a distance matrix. Pass it directly.

```python
# Binary fingerprints → jaccard
TMAP(metric="jaccard").fit(binary_matrix)

# Dense embeddings → cosine or euclidean
TMAP(metric="cosine").fit(embeddings)

# Pre-computed distances
TMAP(metric="precomputed").fit(distance_matrix)
```

## How do I add new points to an existing map?

`add_points()` inserts new points into a fitted model — extends the tree and embedding without refitting.
`transform()` places new points on the map without modifying the model.

```python
model = TMAP().fit(X_train)
model.add_points(X_new)       # mutates model, extends tree
coords = model.transform(X_test)  # read-only projection
```

## How do I find paths or distances in the tree?

```python
path = model.path(idx_a, idx_b)        # list of node indices along tree path
d = model.distance(idx_a, idx_b)       # sum of edge weights
pseudotime = model.distances_from(idx) # tree distance to all other points
```

## How do I load protein sequences?

```python
from tmap.utils import read_fasta, read_protein_csv, read_id_list

ids, seqs = read_fasta("proteins.fa")
ids, seqs = read_protein_csv("data.csv", id_col="accession", seq_col="sequence")
ids = read_id_list("uniprot_ids.txt")
```

## How do I save and load a fitted model?

```python
model.save("my_model.pkl")
model = TMAP.load("my_model.pkl")
```

## What is `kc` and when should I change it?

`kc` controls the number of random augmentation edges added during MST construction. Higher `kc` produces better-connected trees (fewer detached branches) at the cost of longer layout time. Default is 50, which works well for most datasets.

## How do I export to HTML for sharing?

```python
# Quick export
model.to_html("map.html")

# Full control via TmapViz
viz = model.to_tmapviz()
viz.title = "My Map"
viz.add_color_layout("Label", labels, categorical=True)
viz.write_html("map.html")
```

## Binary fingerprints: MinHash or USearch Jaccard?

Both work. MinHash + LSHForest is the default (`metric="jaccard"`). USearch binary Jaccard is an alternative that computes exact Jaccard on packed bit vectors — faster for smaller datasets and guaranteed exact distances. The estimator auto-routes to USearch when it detects dense binary input and the metric is jaccard.